# ESCI 日本語Lexical Search v2

既存の `amazon-jp` を保持したまま、Kuromoji・readingform・文字2–3gramを備えた `amazon-jp-v2` を作成し、全339,059商品をreindexする。Amazon ESCI Task 1日本語test 3,123クエリでv1/v2を比較する。

## 改善施策・チューニング仕様（最初に読む）

### 狙い

v1は全textフィールドがOpenSearch標準analyzerで、日本語の語境界、活用、全半角表記、読み、未知語・複合語への対応が弱かった。v2は **完全・語句一致を壊さず、Kuromojiを主経路、reading/ngramを低boostの回収経路** とする。dense検索ではなくlexical検索だけで、zero-matchと候補順位を改善する段階である。

### Analyzer構成

|analyzer|設定|対象|目的・注意|
|---|---|---|---|
|`ja_text`|`kuromoji_tokenizer(mode=search)` → baseform → part-of-speech除去 → `cjk_width` → `ja_stop` → stemmer → lowercase|title、brand、bullet、description、color|日本語BM25の主経路。複合語を細分化しつつ元語も保持する|
|`ja_reading`|Kuromoji → `kuromoji_readingform(use_romaji=true)` → lowercase|title/brandの`.reading`|漢字・カナとローマ字表記の橋渡し。主経路を上書きしないよう低boost|
|`ja_ngram`|文字2–3gram、letter/digit、`cjk_width`、lowercase|title/brandの`.ngram`|未知語、連結語、タイプミス、型番を回収。長文でノイズとコストが増えるためtitle/brandだけ|
|`lowercase_normalizer`|lowercase + asciifolding|`.raw` keyword|将来の完全一致・filter用。現行query DSLではまだboostに使用していない|

全text analyzerに `html_strip` を入れ、description/bullet内のHTMLタグを索引語から除外する。n-gramをdescriptionへ付けないのは、index膨張と低品質な部分一致を避けるため。

### フィールドと現在のboost

|検索句|boost|意図|
|---|---:|---|
|`match_phrase product_title`|8.0|商品名の語順一致を最優先|
|`match_phrase product_brand`|6.0|ブランドphrase一致|
|`match product_title`|4.0|Kuromojiによるtitle BM25の主スコア|
|`match product_brand`|3.0|ブランド一致|
|`match product_bullet_point`|1.0|仕様・用途の補助|
|`match product_title.reading` / `brand.reading`|各0.8|読み・ローマ字の補助回収|
|`match product_title.ngram`|0.25|titleの部分一致。`minimum_should_match=70%`|
|`match product_brand.ngram`|0.20|brandの部分一致。`minimum_should_match=70%`|
|`match product_description`|0.25|説明文は長くノイズが多いため低boost|

外側は `bool.should` + `minimum_should_match=1`。どれか1経路で一致すれば候補になる。n-gramはcoverage改善用で、phrase/title BM25より順位への影響を小さくしている。

### この初期値で得られた効果

- 公式Task 1 nDCG: **0.8372 → 0.8470**
- zero-matchを0にしたnDCG: **0.8061 → 0.8328**
- zero-match query: **129 → 58**
- 全商品Recall@100: **0.4041 → 0.4845**
- candidate p95 latency: **5ms → 7ms**
- corpus p95 latency: **15.0ms → 58.9ms**
- index容量: **367MB → 476MB（約30%増）**

品質は改善したが、corpus検索ではn-gramによるlatency増が大きい。またquery単位では改善・悪化が混在するため、以下をTask 1 train内validationで調整し、testをチューニングに使わない。

### 次に調整するパラメータ（優先順）

1. **n-gramの発火条件**: query長、英数字・未知語率、Kuromoji候補数に応じてn-gramを使う。通常queryでは外す二段fallbackも比較する。
2. **n-gram boost**: title `0.25`、brand `0.20`をvalidationで探索し、回収改善と順位悪化を分離する。
3. **`minimum_should_match`**: 現在70%。短いqueryでは100%、長いqueryでは70–80%など条件化する。
4. **phrase/title比**: phrase `8`・title `4`が強すぎるqueryを確認し、slopやboostを調整する。
5. **reading boost**: 現在0.8。ローマ字queryでは有効だが、日本語queryで不要な読み一致が順位を乱す場合は条件発火する。
6. **description/bullet**: description `0.25`、bullet `1.0`。用途queryの改善と長文ノイズをvalidationで評価する。
7. **運用コスト**: corpus p95を抑えるため、n-gramをrescore/fallbackへ移す、top-N候補だけ再検索する、gram幅を2だけにする案を比較する。

### 安全性・再実行

既存 `amazon-jp` は変更しない。v2は `amazon-jp-v2`、参照用aliasは `amazon-jp-lexical`。初回reindexは非同期taskを監視し、339,059件一致をassertする。v2が部分投入状態なら自動続行せず停止し、曖昧な上書きを避ける。

In [1]:
from pathlib import Path
import json, math, time
import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown

BASE_URL = "http://localhost:9200"
SOURCE_INDEX = "amazon-jp"
V2_INDEX = "amazon-jp-v2"
V2_ALIAS = "amazon-jp-lexical"
DATA_DIR = Path("esci-data/shopping_queries_dataset")
TIMEOUT = 600
session = requests.Session()
root = session.get(BASE_URL, timeout=30)
root.raise_for_status()
plugins = session.get(f"{BASE_URL}/_cat/plugins?format=json", timeout=30)
plugins.raise_for_status()
assert any(p["component"] == "analysis-kuromoji" for p in plugins.json())
source_count = session.get(f"{BASE_URL}/{SOURCE_INDEX}/_count", timeout=30).json()[
    "count"
]
print(
    f"OpenSearch {root.json()['version']['number']} / source={SOURCE_INDEX}: {source_count:,}"
)

OpenSearch 3.7.0 / source=amazon-jp: 339,059


## 1. Analyzer・mapping設計

In [2]:
index_definition = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "refresh_interval": "-1",
        "analysis": {
            "tokenizer": {
                "ja_search_tokenizer": {"type": "kuromoji_tokenizer", "mode": "search"},
                "ja_ngram_tokenizer": {
                    "type": "ngram",
                    "min_gram": 2,
                    "max_gram": 3,
                    "token_chars": ["letter", "digit"],
                },
            },
            "filter": {
                "ja_reading_romaji": {
                    "type": "kuromoji_readingform",
                    "use_romaji": True,
                },
            },
            "analyzer": {
                "ja_text": {
                    "type": "custom",
                    "char_filter": ["html_strip"],
                    "tokenizer": "ja_search_tokenizer",
                    "filter": [
                        "kuromoji_baseform",
                        "kuromoji_part_of_speech",
                        "cjk_width",
                        "ja_stop",
                        "kuromoji_stemmer",
                        "lowercase",
                    ],
                },
                "ja_reading": {
                    "type": "custom",
                    "char_filter": ["html_strip"],
                    "tokenizer": "ja_search_tokenizer",
                    "filter": ["ja_reading_romaji", "lowercase"],
                },
                "ja_ngram": {
                    "type": "custom",
                    "char_filter": ["html_strip"],
                    "tokenizer": "ja_ngram_tokenizer",
                    "filter": ["cjk_width", "lowercase"],
                },
            },
            "normalizer": {
                "lowercase_normalizer": {
                    "type": "custom",
                    "filter": ["lowercase", "asciifolding"],
                }
            },
        },
    },
    "mappings": {
        "dynamic": "strict",
        "properties": {
            "product_id": {"type": "keyword"},
            "product_title": {
                "type": "text",
                "analyzer": "ja_text",
                "search_analyzer": "ja_text",
                "fields": {
                    "ngram": {
                        "type": "text",
                        "analyzer": "ja_ngram",
                        "search_analyzer": "ja_ngram",
                    },
                    "reading": {
                        "type": "text",
                        "analyzer": "ja_reading",
                        "search_analyzer": "ja_reading",
                    },
                    "raw": {
                        "type": "keyword",
                        "normalizer": "lowercase_normalizer",
                        "ignore_above": 1024,
                    },
                },
            },
            "product_brand": {
                "type": "text",
                "analyzer": "ja_text",
                "search_analyzer": "ja_text",
                "fields": {
                    "ngram": {
                        "type": "text",
                        "analyzer": "ja_ngram",
                        "search_analyzer": "ja_ngram",
                    },
                    "reading": {
                        "type": "text",
                        "analyzer": "ja_reading",
                        "search_analyzer": "ja_reading",
                    },
                    "raw": {
                        "type": "keyword",
                        "normalizer": "lowercase_normalizer",
                        "ignore_above": 512,
                    },
                },
            },
            "product_bullet_point": {
                "type": "text",
                "analyzer": "ja_text",
                "search_analyzer": "ja_text",
            },
            "product_description": {
                "type": "text",
                "analyzer": "ja_text",
                "search_analyzer": "ja_text",
            },
            "product_color": {
                "type": "text",
                "analyzer": "ja_text",
                "search_analyzer": "ja_text",
                "fields": {
                    "raw": {
                        "type": "keyword",
                        "normalizer": "lowercase_normalizer",
                        "ignore_above": 512,
                    },
                },
            },
            "product_locale": {"type": "keyword"},
        },
    },
}
print(json.dumps(index_definition, ensure_ascii=False, indent=2)[:4000])

{
  "settings": {
    "number_of_shards": 1,
    "number_of_replicas": 0,
    "refresh_interval": "-1",
    "analysis": {
      "tokenizer": {
        "ja_search_tokenizer": {
          "type": "kuromoji_tokenizer",
          "mode": "search"
        },
        "ja_ngram_tokenizer": {
          "type": "ngram",
          "min_gram": 2,
          "max_gram": 3,
          "token_chars": [
            "letter",
            "digit"
          ]
        }
      },
      "filter": {
        "ja_reading_romaji": {
          "type": "kuromoji_readingform",
          "use_romaji": true
        }
      },
      "analyzer": {
        "ja_text": {
          "type": "custom",
          "char_filter": [
            "html_strip"
          ],
          "tokenizer": "ja_search_tokenizer",
          "filter": [
            "kuromoji_baseform",
            "kuromoji_part_of_speech",
            "cjk_width",
            "ja_stop",
            "kuromoji_stemmer",
            "lowercase"
          ]
        

## 2. v2作成と全件reindex

In [3]:
exists = session.head(f"{BASE_URL}/{V2_INDEX}", timeout=30).status_code == 200
if not exists:
    r = session.put(f"{BASE_URL}/{V2_INDEX}", json=index_definition, timeout=TIMEOUT)
    r.raise_for_status()
    print("created", V2_INDEX)
else:
    print(V2_INDEX, "already exists; creation skipped")

v2_count = session.get(f"{BASE_URL}/{V2_INDEX}/_count", timeout=30).json()["count"]
if v2_count != source_count:
    if v2_count != 0:
        raise RuntimeError(
            f"{V2_INDEX} has partial data ({v2_count:,}); refusing an ambiguous rerun"
        )
    started = time.time()
    # 長い同期応答は大量のWarning headerを伴うことがあるため、taskを非同期監視する。
    r = session.post(
        f"{BASE_URL}/_reindex?wait_for_completion=false",
        json={
            "conflicts": "abort",
            "source": {"index": SOURCE_INDEX},
            "dest": {"index": V2_INDEX, "op_type": "create"},
        },
        timeout=30,
    )
    r.raise_for_status()
    task_id = r.json()["task"]
    while True:
        active = session.get(
            f"{BASE_URL}/_tasks?actions=*reindex&detailed=false", timeout=30
        )
        active.raise_for_status()
        active_ids = {
            f"{node_id}:{task_num}"
            for node_id, node in active.json().get("nodes", {}).items()
            for task_num in node.get("tasks", {})
        }
        if task_id not in active_ids:
            break
        print(f"reindex running: {time.time()-started:.0f}s", end="\r")
        time.sleep(5)
    print(f"reindex task finished, time={time.time()-started:.1f}s")

session.put(
    f"{BASE_URL}/{V2_INDEX}/_settings",
    json={"index": {"refresh_interval": "1s"}},
    timeout=30,
).raise_for_status()
session.post(f"{BASE_URL}/{V2_INDEX}/_refresh", timeout=30).raise_for_status()
v2_count = session.get(f"{BASE_URL}/{V2_INDEX}/_count", timeout=30).json()["count"]
assert v2_count == source_count == 339059
alias_actions = {"actions": [{"add": {"index": V2_INDEX, "alias": V2_ALIAS}}]}
session.post(f"{BASE_URL}/_aliases", json=alias_actions, timeout=30).raise_for_status()
print(f"{V2_INDEX}: {v2_count:,} docs / alias={V2_ALIAS}")

amazon-jp-v2 already exists; creation skipped
amazon-jp-v2: 339,059 docs / alias=amazon-jp-lexical


In [4]:
analyzer_examples = []
for text in ["ワイヤレス充電器", "ミニターボファン", "Nintendo６４", "花譜"]:
    for analyzer in ["ja_text", "ja_reading", "ja_ngram"]:
        r = session.post(
            f"{BASE_URL}/{V2_INDEX}/_analyze",
            json={"analyzer": analyzer, "text": text},
            timeout=30,
        )
        r.raise_for_status()
        tokens = [x["token"] for x in r.json()["tokens"]]
        analyzer_examples.append(
            {"text": text, "analyzer": analyzer, "tokens": " | ".join(tokens[:20])}
        )
display(pd.DataFrame(analyzer_examples))

,text,analyzer,tokens
0,ワイヤレス充電器,ja_text,ワイヤレス | 充電 | 器
1,ワイヤレス充電器,ja_reading,waiyaresu | jūden | ki
2,ワイヤレス充電器,ja_ngram,ワイ | ワイヤ | イヤ | イヤレ | ヤレ | ヤレス | レス | レス充 | ス充...
3,ミニターボファン,ja_text,ミニ | ミニターボファン | ターボ | ファン
4,ミニターボファン,ja_reading,mini | minitabofan | tabo | fan
5,ミニターボファン,ja_ngram,ミニ | ミニタ | ニタ | ニター | ター | ターボ | ーボ | ーボフ | ボフ...
6,Nintendo６４,ja_text,nintendo | 6 | 4
7,Nintendo６４,ja_reading,nintendo | roku | yon
8,Nintendo６４,ja_ngram,ni | nin | in | int | nt | nte | te | ten | en...
9,花譜,ja_text,花 | 譜


## 実行結果まとめ（2026-06-20）

|公式candidate評価|v1|v2|差分|
|---|---:|---:|---:|
|official nDCG|0.8372|**0.8470**|+0.0098|
|zero-matchを0にしたnDCG|0.8061|**0.8328**|+0.0267|
|nDCG@10|0.6913|**0.7108**|+0.0195|
|candidate match率|0.7633|**0.7700**|+0.0067|
|zero-match query|129|**58**|-71|
|candidate latency p95|5.0ms|7.0ms|+2.0ms|

全商品検索でもnDCG@10は0.3528→0.3935、Recall@100は0.4041→0.4845、zero-hitは89→20へ改善した。一方でcorpus検索p95は15.0ms→58.9ms、index容量は367MB→476MB（約30%増）。n-gramのcoverage効果は大きいが、全商品検索ではlatency最適化が次の課題。

query単位では大きな改善と悪化が混在する。testを見てboostを調整すると過学習になるため、次の重み調整はTask 1 train内のquery単位validationで行い、testは最終確認にのみ使用する。

## 3. v1/v2検索クエリ

In [5]:
def v1_query(query):
    return {
        "multi_match": {
            "query": query,
            "type": "best_fields",
            "operator": "or",
            "fields": [
                "product_title^4",
                "product_brand^2",
                "product_bullet_point",
                "product_description^0.5",
            ],
        }
    }


def v2_query(query):
    return {
        "bool": {
            "minimum_should_match": 1,
            "should": [
                {"match_phrase": {"product_title": {"query": query, "boost": 8}}},
                {
                    "match": {
                        "product_title": {"query": query, "boost": 4, "operator": "or"}
                    }
                },
                {"match_phrase": {"product_brand": {"query": query, "boost": 6}}},
                {
                    "match": {
                        "product_brand": {"query": query, "boost": 3, "operator": "or"}
                    }
                },
                {
                    "match": {
                        "product_title.reading": {
                            "query": query,
                            "boost": 0.8,
                            "operator": "or",
                        }
                    }
                },
                {
                    "match": {
                        "product_brand.reading": {
                            "query": query,
                            "boost": 0.8,
                            "operator": "or",
                        }
                    }
                },
                {
                    "match": {
                        "product_title.ngram": {
                            "query": query,
                            "boost": 0.25,
                            "minimum_should_match": "70%",
                        }
                    }
                },
                {
                    "match": {
                        "product_brand.ngram": {
                            "query": query,
                            "boost": 0.2,
                            "minimum_should_match": "70%",
                        }
                    }
                },
                {"match": {"product_bullet_point": {"query": query, "boost": 1}}},
                {"match": {"product_description": {"query": query, "boost": 0.25}}},
            ],
        }
    }


display(
    pd.DataFrame(
        [
            {
                "variant": "v1",
                "query": json.dumps(v1_query("ワイヤレス充電器"), ensure_ascii=False),
            },
            {
                "variant": "v2",
                "query": json.dumps(v2_query("ワイヤレス充電器"), ensure_ascii=False),
            },
        ]
    )
)

,variant,query
0,v1,"{""multi_match"": {""query"": ""ワイヤレス充電器"", ""type"": ..."
1,v2,"{""bool"": {""minimum_should_match"": 1, ""should"":..."


## 4. Amazon ESCI Task 1日本語testによる比較

In [6]:
examples = pd.read_parquet(
    DATA_DIR / "shopping_queries_dataset_examples.parquet",
    filters=[
        ("product_locale", "=", "jp"),
        ("split", "=", "test"),
        ("small_version", "=", 1),
    ],
).sort_values(["query_id", "example_id"])
groups = {qid: g for qid, g in examples.groupby("query_id", sort=True)}
assert len(groups) == 3123 and len(examples) == 88789
OFFICIAL_GAIN = {"E": 1.0, "C": 0.1, "S": 0.01, "I": 0.0}


def dcg(labels, k=None):
    labels = labels if k is None else labels[:k]
    return sum(OFFICIAL_GAIN[x] / math.log2(i + 2) for i, x in enumerate(labels))


def ndcg(run_ids, qrels, k=None):
    ideal = sorted(qrels.values(), key=OFFICIAL_GAIN.get, reverse=True)
    denom = dcg(ideal, k)
    return dcg([qrels.get(pid, "I") for pid in run_ids], k) / denom if denom else 0.0


def candidate_body(query, ids, query_dsl):
    return {
        "size": len(ids),
        "_source": False,
        "query": {
            "bool": {
                "must": [query_dsl],
                "filter": [{"terms": {"product_id": ids}}],
            }
        },
    }


specs = []
for qid, g in groups.items():
    query = g["query"].iloc[0]
    ids = g.product_id.tolist()
    for variant, index, qdsl in [
        ("v1", SOURCE_INDEX, v1_query(query)),
        ("v2", V2_INDEX, v2_query(query)),
    ]:
        specs.append(
            ((qid, variant, "candidate"), index, candidate_body(query, ids, qdsl))
        )
        specs.append(
            (
                (qid, variant, "corpus"),
                index,
                {"size": 100, "_source": False, "query": qdsl},
            )
        )


def run_msearch(specs, batch_size=200):
    outputs = {}
    took = {}
    for start in range(0, len(specs), batch_size):
        batch = specs[start : start + batch_size]
        lines = []
        for key, index, body in batch:
            lines.extend(
                [json.dumps({"index": index}), json.dumps(body, ensure_ascii=False)]
            )
        r = session.post(
            f"{BASE_URL}/_msearch",
            headers={"Content-Type": "application/x-ndjson"},
            data=("\n".join(lines) + "\n").encode(),
            timeout=TIMEOUT,
        )
        r.raise_for_status()
        responses = r.json()["responses"]
        for (key, _, _), response in zip(batch, responses):
            if "error" in response:
                raise RuntimeError(f'{key}: {response["error"]}')
            outputs[key] = response["hits"]["hits"]
            took[key] = response["took"]
        if start == 0 or start + len(batch) == len(specs) or start % 2000 == 0:
            print(f"{start + len(batch):,}/{len(specs):,}")
    return outputs, took


started = time.time()
outputs, took = run_msearch(specs)
print(f"evaluation searches={len(specs):,}, wall={time.time()-started:.1f}s")

200/12,492


2,200/12,492


4,200/12,492


6,200/12,492


8,200/12,492


10,200/12,492


12,200/12,492
12,492/12,492
evaluation searches=12,492, wall=6.0s


In [7]:
candidate_rows = []
corpus_rows = []
for qid, g in groups.items():
    qrels = dict(zip(g.product_id, g.esci_label))
    ids = list(qrels)
    positive = {pid for pid, label in qrels.items() if OFFICIAL_GAIN[label] > 0}
    for variant in ["v1", "v2"]:
        hits = outputs[(qid, variant, "candidate")]
        scores = {h["_id"]: h["_score"] for h in hits}
        ranked = sorted(
            ids, key=lambda pid: (pid not in scores, -scores.get(pid, 0), pid)
        )
        candidate_rows.append(
            {
                "query_id": qid,
                "query": g["query"].iloc[0],
                "variant": variant,
                "official_nDCG": ndcg(ranked, qrels),
                "nDCG_zero_match_0": 0 if not scores else ndcg(ranked, qrels),
                "nDCG@10": ndcg(ranked, qrels, 10),
                "match_rate": len(scores) / len(ids),
                "zero_match": not scores,
                "took_ms": took[(qid, variant, "candidate")],
            }
        )
        run_ids = [h["_id"] for h in outputs[(qid, variant, "corpus")]]
        corpus_rows.append(
            {
                "query_id": qid,
                "variant": variant,
                "nDCG@10_unjudged0": ndcg(run_ids[:10], qrels, 10),
                "judged@10": len(set(run_ids[:10]) & set(qrels)) / 10,
                "Recall@100": len(set(run_ids) & positive) / len(positive),
                "zero_hit": not run_ids,
                "took_ms": took[(qid, variant, "corpus")],
            }
        )
candidate_eval = pd.DataFrame(candidate_rows)
corpus_eval = pd.DataFrame(corpus_rows)
candidate_summary = (
    candidate_eval.groupby("variant")
    .agg(
        official_nDCG=("official_nDCG", "mean"),
        nDCG_zero_match_0=("nDCG_zero_match_0", "mean"),
        nDCG_at_10=("nDCG@10", "mean"),
        match_rate=("match_rate", "mean"),
        zero_match=("zero_match", "sum"),
        latency_p50_ms=("took_ms", "median"),
        latency_p95_ms=("took_ms", lambda s: s.quantile(0.95)),
    )
    .reset_index()
)
corpus_summary = (
    corpus_eval.groupby("variant")
    .agg(
        nDCG_at_10_unjudged0=("nDCG@10_unjudged0", "mean"),
        judged_at_10=("judged@10", "mean"),
        Recall_at_100=("Recall@100", "mean"),
        zero_hit=("zero_hit", "sum"),
        latency_p50_ms=("took_ms", "median"),
        latency_p95_ms=("took_ms", lambda s: s.quantile(0.95)),
    )
    .reset_index()
)
display(Markdown("### 公式candidate reranking（主評価）"))
display(candidate_summary.round(4))
display(Markdown("### 全商品検索（未判定=0の参考値）"))
display(corpus_summary.round(4))

### 公式candidate reranking（主評価）

,variant,official_nDCG,nDCG_zero_match_0,nDCG_at_10,match_rate,zero_match,latency_p50_ms,latency_p95_ms
0,v1,0.8372,0.8061,0.6913,0.7633,129,1.0,4.0
1,v2,0.8470,0.8328,0.7108,0.7700,58,1.0,5.0


### 全商品検索（未判定=0の参考値）

,variant,nDCG_at_10_unjudged0,judged_at_10,Recall_at_100,zero_hit,latency_p50_ms,latency_p95_ms
0,v1,0.3528,0.3734,0.4041,89,3.0,11.0
1,v2,0.3935,0.4256,0.4845,20,4.0,10.0


In [8]:
pivot = candidate_eval.pivot(
    index=["query_id", "query"], columns="variant", values="official_nDCG"
).reset_index()
pivot["delta_v2-v1"] = pivot["v2"] - pivot["v1"]
display(Markdown("### v2改善上位15"))
display(pivot.nlargest(15, "delta_v2-v1").round(4))
display(Markdown("### v2悪化上位15"))
display(pivot.nsmallest(15, "delta_v2-v1").round(4))
index_stats = session.get(
    f"{BASE_URL}/_cat/indices/{SOURCE_INDEX},{V2_INDEX}?format=json&bytes=mb",
    timeout=30,
).json()
display(Markdown("### インデックス容量"))
display(
    pd.DataFrame(index_stats)[["index", "docs.count", "store.size", "health", "status"]]
)

### v2改善上位15

variant,query_id,query,v1,v2,delta_v2-v1
1271,119801,サーモンチップ,0.3599,0.9998,0.6399
1311,120058,ジェイソンボーン,0.3738,0.9985,0.6247
1121,118781,ガン ロッカー,0.3735,0.9884,0.6149
2506,127301,日本ブロアー,0.3934,1.0000,0.6066
2640,127895,江戸の飛脚,0.3884,0.9924,0.6040
1472,121153,テトラat50 フィルター,0.3667,0.9493,0.5825
1949,124131,メルくん,0.4996,0.9964,0.4969
545,100372,taeccl 携帯扇風機 羽根なし 小型扇風機,0.4590,0.9249,0.4659
1327,120172,ジョニーイングリッシュ,0.5377,0.9989,0.4612
1462,121079,チャークロス,0.4362,0.8889,0.4526


### v2悪化上位15

variant,query_id,query,v1,v2,delta_v2-v1
1866,123661,マイクロビキニ 大きいサイズ,0.9992,0.4588,-0.5404
1111,118702,カード デザイン,0.9601,0.4968,-0.4633
21,3197,2dｓllセット,0.9996,0.5534,-0.4461
2164,125468,伊達 扇子,0.9936,0.5622,-0.4314
1516,121376,トキオ ベスト,0.9581,0.5343,-0.4239
519,96316,sound warrior sw-hp10s,0.8530,0.4468,-0.4062
1993,124464,ラッシュ ガード 厚手,0.8242,0.4247,-0.3994
1265,119761,サンレレ,1.0000,0.6300,-0.3700
1661,122381,パワーストリップ,0.9451,0.5905,-0.3546
1717,122718,フィルム lgv20,0.9770,0.6259,-0.3511


### インデックス容量

,index,docs.count,store.size,health,status
0,amazon-jp,339059,367,yellow,open
1,amazon-jp-v2,339059,476,green,open


## 5–6. train内validation tuningとheld-out test

Task 1 JP trainをquery単位のSHA-1 hashで80/20に固定分割し、14設定をvalidationで比較する。公式nDCG最大、zero-match非悪化、p95制約内の設定を自動選択する。testは選択後に一度だけ評価し、defaultとの差をpaired bootstrap 95% CI付きで報告する。下のセルは上部に保持した非表示のtuning stageを順番に実行する。

In [9]:
import nbformat
self_notebook = nbformat.read('esci_lexical_v2.ipynb', as_version=4)
stages = sorted(
    (c.metadata['tuning_stage'], c) for c in self_notebook.cells
    if c.cell_type == 'raw' and 'tuning_stage' in c.metadata
)
assert [stage for stage, _ in stages] == [1, 2, 3, 4]
for stage, cell in stages:
    print(f'\n--- tuning stage {stage} ---')
    exec(compile(cell.source, f'tuning-stage-{stage}', 'exec'), globals())


--- tuning stage 1 ---


,Task 1 JP train queries,fit queries (reserved),validation queries,validation judgements,test queries used for tuning
0,7284,5844,1440,41026,0



--- tuning stage 2 ---


,phrase_title,phrase_brand,title,brand,reading,ngram_title,ngram_brand,ngram_msm,bullet,description
default,8.0,6.0,4.0,3.0,0.8,0.25,0.2,70%,1.0,0.25
no_reading,8.0,6.0,4.0,3.0,0.0,0.25,0.2,70%,1.0,0.25
reading_0.4,8.0,6.0,4.0,3.0,0.4,0.25,0.2,70%,1.0,0.25
no_ngram,8.0,6.0,4.0,3.0,0.8,0.0,0.0,70%,1.0,0.25
ngram_low_80,8.0,6.0,4.0,3.0,0.8,0.1,0.08,80%,1.0,0.25
ngram_low_100,8.0,6.0,4.0,3.0,0.8,0.1,0.08,100%,1.0,0.25
ngram_default_80,8.0,6.0,4.0,3.0,0.8,0.25,0.2,80%,1.0,0.25
ngram_default_100,8.0,6.0,4.0,3.0,0.8,0.25,0.2,100%,1.0,0.25
phrase_6_4,6.0,4.0,4.0,3.0,0.8,0.25,0.2,70%,1.0,0.25
phrase_10_8,10.0,8.0,4.0,3.0,0.8,0.25,0.2,70%,1.0,0.25



--- tuning stage 3 ---


...

### held-out test: candidate

,variant,official_nDCG,nDCG_zero_match_0,nDCG_at_10,match_rate,zero_match,latency_p50_ms,latency_p95_ms
0,v2_default,0.84701,0.83285,0.71077,0.77002,58,1.0,5.0
1,v2_tuned:content_low,0.84744,0.83328,0.71158,0.77002,58,1.0,4.0


### held-out test: corpus参考値

,variant,nDCG_at_10_unjudged0,judged_at_10,Recall_at_100,zero_hit,latency_p50_ms,latency_p95_ms
0,v2_default,0.39349,0.42565,0.48452,20,4.0,10.0
1,v2_tuned:content_low,0.39419,0.42533,0.48468,20,4.0,10.0


paired mean nDCG delta=0.000429, bootstrap 95% CI=[-0.000348, 0.001147]
wins=886, ties=1417, losses=820


## Boost tuning結果

train 7,284 queryのうちSHA-1 hashで固定した1,440 query（41,026 judgments）をvalidationに使用し、14設定を比較した。残る5,844 queryは将来のモデル学習用に予約し、testは設定選択に使用していない。

### 選択設定: `content_low`

変更は次の2点だけで、phrase/title/brand/reading/ngramは初期v2を維持する。

- `product_bullet_point`: **1.0 → 0.5**
- `product_description`: **0.25 → 0.10**

validation公式nDCGはdefault 0.83945から **0.83966**。zero-match、match率、p95は同等で、長い補助テキストのノイズを弱める設定が僅差で最良だった。

|held-out test|v2 default|`content_low`|差分|
|---|---:|---:|---:|
|公式nDCG|0.84701|**0.84744**|+0.00043|
|zero-matchを0にしたnDCG|0.83285|**0.83328**|+0.00043|
|nDCG@10|0.71077|**0.71158**|+0.00081|
|zero-match|58|58|0|

query単位では886勝・1,417同順位・820敗。公式nDCG差のpaired bootstrap 95% CIは **[-0.00035, 0.00115]** でゼロを跨ぐ。したがって`content_low`を暫定推奨とするが、強い改善証拠ではない。主な成果は初期v2のanalyzer/ngram導入であり、boost微調整の増分は小さい。次の大きな改善にはLTRまたはquery条件付きfallbackが必要。

実装上は `tuned_query(query, best_config)` を利用する。test結果を見て14候補へ戻って再選択はしない。

## 判定基準

v2を採用する条件は、公式nDCGとzero-match保守nDCGが改善し、zero-match queryが減少すること。p95 latencyと容量増加が大きい場合はn-gram boost・対象フィールド・gram幅を調整する。既存 `amazon-jp` は変更しておらず、alias `amazon-jp-lexical` のみv2を指す。